# Focus versus AR(1): rolling IPCA forecasts at 3, 6, and 12 months

**Question.** On the same historical forecast months, how do constructed Focus forecasts compare with a simple AR(1) rule fitted to realized monthly IPCA?

The AR(1) rule is fitted separately at each forecast origin, using only monthly IPCA rates available through the preceding month. It recursively predicts the next 12 monthly rates and compounds the first 3, 6, or 12. The first forecast uses 60 observed months for training, so the common comparison runs from **November 2006 through January 2025**. Focus forecasts and realized cumulative IPCA come from the existing matched panel. All errors are in percentage points.

Inputs: [local data rebuild instructions](data/README.md), [Focus horizon comparison](ipca_rolling_horizon_comparison.ipynb), and [IBGE IPCA index](https://sidra.ibge.gov.br/tabela/1737).


## 1. Load and validate the matched data

Run from the repository root or from projects/. The matched panel is generated by the Focus horizon comparison notebook; the IBGE index is downloaded by projects/download_ibge_ipca.R.


In [ ]:
panel_file <- file.path("projects", "data", "ipca_rolling_horizon_comparison_200111_202501.csv")
project_root <- if (file.exists(panel_file)) {
  "."
} else if (file.exists(file.path("..", panel_file))) {
  ".."
} else {
  stop("Missing matched Focus panel; run ipca_rolling_horizon_comparison.ipynb first")
}
panel <- read.csv(file.path(project_root, panel_file), stringsAsFactors = FALSE)
ibge <- read.csv(file.path(project_root, "projects/data/ibge_ipca_monthly_index_200110_202512.csv"),
                 colClasses = c("character", "numeric"))
expected_panel <- format(seq(as.Date("2001-11-01"), as.Date("2025-01-01"), by = "month"), "%Y-%m")
expected_index <- format(seq(as.Date("2001-10-01"), as.Date("2025-12-01"), by = "month"), "%Y%m")
stopifnot(identical(panel$forecast_month, expected_panel),
          identical(ibge$month, expected_index),
          !anyNA(ibge$ipca_index), all(ibge$ipca_index > 0),
          all(panel$focus_snapshot == panel$focus_snapshot[1]))
index_anchors <- match(gsub("-", "", panel$forecast_month), ibge$month)
stopifnot(!anyNA(index_anchors))
for (h in c(3L, 6L, 12L)) {
  realized_from_index <- 100 * (
    ibge$ipca_index[index_anchors + h - 1L] /
      ibge$ipca_index[index_anchors - 1L] - 1
  )
  if (max(abs(realized_from_index - panel[[paste0("realized_", h, "m_pct")]])) > 1e-8) {
    stop("IBGE index and matched panel disagree; rebuild the matched panel")
  }
}
monthly <- data.frame(
  month = ibge$month[-1],
  ipca_pct = 100 * (ibge$ipca_index[-1] / ibge$ipca_index[-nrow(ibge)] - 1)
)
stopifnot(all(is.finite(monthly$ipca_pct)))
cat("Focus snapshot:", panel$focus_snapshot[1], "| monthly IPCA rates:", nrow(monthly), "\n")


## 2. Fit an expanding-window AR(1) forecast at each origin

For monthly IPCA inflation, use the conditional least-squares rule y(t) = alpha + phi × y(t−1) + innovation. At the end of forecast month M, the latest assumed available IPCA observation is for M−1. A 60-month initial history starts the common sample in November 2006. Neither model fitting nor horizon construction uses realized inflation from M onward.


In [ ]:
horizons <- c(3L, 6L, 12L)
training_months <- 60L
anchor_positions <- match(gsub("-", "", panel$forecast_month), monthly$month)
stopifnot(!anyNA(anchor_positions))
usable <- which(anchor_positions > training_months)
comparison <- panel[usable, , drop = FALSE]
row.names(comparison) <- NULL
stopifnot(identical(comparison$forecast_month[1], "2006-11"),
          identical(tail(comparison$forecast_month, 1), "2025-01"))

for (h in horizons) {
  comparison[[paste0("ar1_forecast_", h, "m_pct")]] <- NA_real_
  comparison[[paste0("ar1_error_", h, "m_pp")]] <- NA_real_
  comparison[[paste0("ar1_absolute_error_", h, "m_pp")]] <- NA_real_
}
comparison$ar1_alpha <- NA_real_
comparison$ar1_phi <- NA_real_

for (i in seq_len(nrow(comparison))) {
  position <- anchor_positions[usable[i]]
  history <- monthly$ipca_pct[seq_len(position - 1L)]
  stopifnot(length(history) >= training_months)
  fit <- lm(current ~ previous,
            data = data.frame(current = history[-1L],
                              previous = history[-length(history)]))
  coefficients <- coef(fit)
  if (any(!is.finite(coefficients))) stop("Invalid AR(1) fit at ", comparison$forecast_month[i])
  comparison$ar1_alpha[i] <- coefficients[1]
  comparison$ar1_phi[i] <- coefficients[2]

  predicted_months <- numeric(max(horizons))
  previous <- tail(history, 1)
  for (j in seq_along(predicted_months)) {
    predicted_months[j] <- coefficients[1] + coefficients[2] * previous
    previous <- predicted_months[j]
  }
  if (any(1 + predicted_months / 100 <= 0)) stop("Invalid compounded forecast")
  cumulative <- 100 * (cumprod(1 + predicted_months / 100)[horizons] - 1)
  for (k in seq_along(horizons)) {
    h <- horizons[k]
    forecast_name <- paste0("ar1_forecast_", h, "m_pct")
    error_name <- paste0("ar1_error_", h, "m_pp")
    absolute_name <- paste0("ar1_absolute_error_", h, "m_pp")
    comparison[[forecast_name]][i] <- cumulative[k]
    comparison[[error_name]][i] <- cumulative[k] - comparison[[paste0("realized_", h, "m_pct")]][i]
    comparison[[absolute_name]][i] <- abs(comparison[[error_name]][i])
  }
}
stopifnot(nrow(comparison) == 219L,
          all(is.finite(comparison$ar1_phi)),
          all(vapply(comparison[grep("^ar1_absolute_error_", names(comparison))],
                     function(x) all(is.finite(x)), logical(1))))
write.csv(comparison, file.path(project_root, "projects/data/ipca_focus_vs_ar1_200611_202501.csv"),
          row.names = FALSE)
cat("Common forecast months:", nrow(comparison), "| first:", comparison$forecast_month[1],
    "| last:", tail(comparison$forecast_month, 1), "\n")


## 3. Compare forecast accuracy

The table reports both models on exactly the same 219 forecast months at each horizon. “Focus gain” is AR(1) MAE minus Focus MAE, so a positive value favors Focus. The absolute errors are cumulative over different window lengths; compare the two methods **within** each horizon.


In [ ]:
accuracy <- do.call(rbind, lapply(horizons, function(h) {
  focus_error <- comparison[[paste0("error_", h, "m_pp")]]
  ar1_error <- comparison[[paste0("ar1_error_", h, "m_pp")]]
  focus_mae <- mean(abs(focus_error))
  ar1_mae <- mean(abs(ar1_error))
  data.frame(horizon_months = h, n = length(focus_error),
             focus_mae_pp = focus_mae, ar1_mae_pp = ar1_mae,
             focus_gain_pp = ar1_mae - focus_mae,
             focus_relative_gain_pct = 100 * (1 - focus_mae / ar1_mae),
             focus_bias_pp = mean(focus_error), ar1_bias_pp = mean(ar1_error),
             focus_rmse_pp = sqrt(mean(focus_error^2)),
             ar1_rmse_pp = sqrt(mean(ar1_error^2)))
}))
print(accuracy, row.names = FALSE, digits = 4)

options(repr.plot.width = 10, repr.plot.height = 6)
par(bg = "white")
mae_matrix <- rbind(Focus = accuracy$focus_mae_pp, AR1 = accuracy$ar1_mae_pp)
barplot(mae_matrix, beside = TRUE, col = c("steelblue4", "darkorange2"),
        names.arg = paste0(horizons, " months"),
        ylim = c(0, max(mae_matrix) * 1.18),
        ylab = "Mean absolute error (p.p.)",
        main = "Focus and AR(1) accuracy on matched months")
legend("topleft", legend = rownames(mae_matrix),
       fill = c("steelblue4", "darkorange2"), bty = "n")


**Validated result on 219 matched months (November 2006–January 2025):**

| Horizon | Focus MAE | AR(1) MAE | Focus gain | Focus RMSE | AR(1) RMSE |
| --- | ---: | ---: | ---: | ---: | ---: |
| 3 months | 0.385 | 0.589 | 0.205 | 0.521 | 0.759 |
| 6 months | 0.835 | 1.121 | 0.285 | 1.119 | 1.403 |
| 12 months | 1.532 | 1.790 | 0.258 | 2.220 | 2.194 |

All error measures are in percentage points. Focus has lower MAE at each horizon; AR(1) has slightly lower RMSE at 12 months.

![Mean absolute error for Focus and AR(1) at three horizons](figures/ipca_focus_vs_ar1_mae.png)


## 4. Compare the full error distributions

Empirical CDFs show the fraction of forecast months with absolute error below each threshold. A curve farther left indicates more small errors for that horizon.


In [ ]:
options(repr.plot.width = 12, repr.plot.height = 4)
par(mfrow = c(1, 3), mar = c(4, 4, 3, 1), bg = "white")
for (h in horizons) {
  focus_absolute <- comparison[[paste0("absolute_error_", h, "m_pp")]]
  ar1_absolute <- comparison[[paste0("ar1_absolute_error_", h, "m_pp")]]
  x_max <- max(c(focus_absolute, ar1_absolute))
  plot(ecdf(focus_absolute), do.points = FALSE, verticals = TRUE,
       col = "steelblue4", lwd = 2, xlim = c(0, x_max),
       xlab = "Absolute error (p.p.)", ylab = "Empirical probability",
       main = paste(h, "months"))
  lines(ecdf(ar1_absolute), do.points = FALSE, verticals = TRUE,
        col = "darkorange2", lwd = 2)
  legend("bottomright", c("Focus", "AR(1)"),
         col = c("steelblue4", "darkorange2"), lty = 1, lwd = 2, bty = "n")
}
par(mfrow = c(1, 1))


![Empirical CDFs of Focus and AR(1) absolute errors](figures/ipca_focus_vs_ar1_ecdf.png)

At 12 months, the ECDFs cross: Focus has more small errors, while some large Focus errors affect RMSE.


## 5. Quantify paired uncertainty

A circular moving-block bootstrap resamples whole monthly rows, preserving the pairing of Focus and AR(1) errors and some dependence from overlapping outcome windows. Use 12-month blocks, 10,000 draws, and percentile intervals for Focus's MAE gain at each horizon.


In [ ]:
set.seed(20261106)
B <- 10000L
block_length <- 12L
n <- nrow(comparison)
gain_matrix <- sapply(horizons, function(h) {
  comparison[[paste0("ar1_absolute_error_", h, "m_pp")]] -
    comparison[[paste0("absolute_error_", h, "m_pp")]]
})
colnames(gain_matrix) <- paste0(horizons, " months")
boot_gain <- matrix(NA_real_, nrow = B, ncol = length(horizons))
for (b in seq_len(B)) {
  starts <- sample.int(n, ceiling(n / block_length), replace = TRUE)
  idx <- ((rep(starts - 1L, each = block_length) +
           rep(0:(block_length - 1L), times = length(starts))) %% n) + 1L
  boot_gain[b, ] <- colMeans(gain_matrix[idx[seq_len(n)], , drop = FALSE])
}
stopifnot(all(is.finite(boot_gain)))
intervals <- apply(boot_gain, 2, quantile, probs = c(0.025, 0.975))
bootstrap_summary <- data.frame(
  horizon_months = horizons,
  focus_gain_pp = colMeans(gain_matrix),
  lower_95_pp = intervals[1, ],
  upper_95_pp = intervals[2, ],
  fraction_focus_better = colMeans(boot_gain > 0)
)
print(bootstrap_summary, row.names = FALSE, digits = 4)
convergence <- data.frame(
  draws = c(1000L, 5000L, B),
  twelve_month_lower_95 = vapply(c(1000L, 5000L, B), function(k)
    unname(quantile(boot_gain[seq_len(k), 3], 0.025)), numeric(1)),
  twelve_month_upper_95 = vapply(c(1000L, 5000L, B), function(k)
    unname(quantile(boot_gain[seq_len(k), 3], 0.975)), numeric(1))
)
print(convergence, row.names = FALSE, digits = 4)


**Paired 12-month-block bootstrap (10,000 draws):**

| Horizon | Focus MAE gain | 95% interval | Fraction of draws favoring Focus |
| --- | ---: | ---: | ---: |
| 3 months | 0.205 p.p. | 0.136 to 0.271 | 1.000 |
| 6 months | 0.285 p.p. | 0.092 to 0.454 | 0.998 |
| 12 months | 0.258 p.p. | −0.204 to 0.681 | 0.870 |

The 12-month interval includes zero; the historical MAE advantage at that horizon is less certain.


### Interpretation

The AR(1) rule is a deliberately simple benchmark: it uses only lagged realized monthly IPCA. Focus includes additional information available to survey respondents. The AR(1) fit is updated every month, but coefficient instability and economic regime changes may affect it. Bootstrap intervals summarize uncertainty in this historical comparison; they do not prove future performance.
